In [1]:
# Đoạn code này tự động lấy Tên tài khoản và Tên máy tính
import getpass
import socket
user = getpass.getuser()
host = socket.gethostname()
def get_system_context():
    user = getpass.getuser()
    host = socket.gethostname()
    return f"[USER: {user}] [HOST: {host}]"

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [3]:
listings_df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .load("../data/Listings.csv")
    # .load("hdfs://master-node:9000/AirbnbData/Listings.csv")  # dùng khi master-node bật
listings_df_raw.show(truncate=False)

+----------+---------------------------------------------------+--------+----------+----------------------------+------------------+------------------+--------------------+-----------------+-------------------------+--------------------+----------------------+-------------------+--------+-----+--------+---------+----------------+------------+------------+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+--------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+----------------+
|listing_id|na

In [4]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import ArrayType, StringType
listings_df = listings_df_raw.withColumn(
    "amenities",
    from_json(col("amenities"), ArrayType(StringType()))
)

In [5]:
from pyspark.sql.functions import when, col
# Quy đổi toàn bộ giá từ tiền tệ địa phương (Local Currency) sang USD
# Tỷ giá được cập nhật theo thời điểm hiện tại
listings_df_currency = listings_df.withColumn(
    "price_usd",
    when(col("city") == "New York", col("price") * 1.0)           # USD giữ nguyên
    .when(col("city") == "Paris", col("price") * 1.156)            # EUR -> USD
    .when(col("city") == "Rome", col("price") * 1.156)             # EUR -> USD
    .when(col("city") == "Sydney", col("price") * 0.7038)           # AUD -> USD
    .when(col("city") == "Hong Kong", col("price") * 0.1276)        # HKD -> USD
    .when(col("city") == "Bangkok", col("price") * 0.03048)         # THB -> USD
    .when(col("city") == "Mexico City", col("price") * 0.05795)     # MXN -> USD
    .when(col("city") == "Cape Town", col("price") * 0.06130)       # ZAR -> USD
    .when(col("city") == "Istanbul", col("price") *  0.02162)        # TRY -> USD
    .when(col("city") == "Rio de Janeiro", col("price") * 0.1961 )   # BRL -> USD
    .otherwise(col("price"))
)

In [6]:
listings_df_currency.show()

+----------+--------------------+--------+----------+--------------------+------------------+------------------+--------------------+-----------------+-------------------------+--------------------+----------------------+-------------------+--------+-----+--------+---------+----------------+------------+------------+--------+--------------------+-----+--------------+--------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+----------------+------------------+
|listing_id|                name| host_id|host_since|       host_location|host_response_time|host_response_rate|host_acceptance_rate|host_is_superhost|host_total_listings_count|host_has_profile_pic|host_identity_verified|      neighbourhood|district| city|latitude|longitude|   property_type|   room_type|accommodates|bedrooms|           amenities|price|minimum_nights|maximum_nights|review_scores_rating|review_scor

In [7]:
listings_df_currency.printSchema()

root
 |-- listing_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: double (nullable = true)
 |-- host_acceptance_rate: double (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_total_listings_count: integer (nullable = true)
 |-- host_has_profile_pic: string (nullable = true)
 |-- host_identity_verified: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- district: string (nullable = true)
 |-- city: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- amenities: array (nullable = true)
 |    |-- el

In [8]:
reviews_df=spark.read.csv("../data/Reviews.csv",header=True,inferSchema=True)
# reviews_df=spark.read.csv("hdfs://master-node:9000/AirbnbData/Reviews.csv",header=True,inferSchema=True)  # dùng khi master-node bật
reviews_df.show()

+----------+---------+----------+-----------+
|listing_id|review_id|      date|reviewer_id|
+----------+---------+----------+-----------+
|     11798|330265172|2018-09-30|   11863072|
|     15383|330103585|2018-09-30|   39147453|
|     16455|329985788|2018-09-30|    1125378|
|     17919|330016899|2018-09-30|  172717984|
|     26827|329995638|2018-09-30|   17542859|
|     74561|330089224|2018-09-30|  173044789|
|    140355|330194958|2018-09-30|  160093807|
|    162163|329980859|2018-09-30|   94026758|
|    167998|329950677|2018-09-30|   35388162|
|    178188|330213008|2018-09-30|    3652511|
|    232956|330140226|2018-09-30|   57664145|
|    266725|330038354|2018-09-30|   77387165|
|    314288|330299075|2018-09-30|  192717587|
|    325880|330081449|2018-09-30|  154527568|
|    335003|329968377|2018-09-30|    3461699|
|    348747|330131287|2018-09-30|    9554201|
|    352851|330201364|2018-09-30|  142182690|
|    378714|330246144|2018-09-30|   15772951|
|    406852|330283854|2018-09-30| 

In [10]:
reviews_df.printSchema()

root
 |-- listing_id: integer (nullable = true)
 |-- review_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)



In [9]:
listings_df_currency.createOrReplaceTempView("listings")
reviews_df.createOrReplaceTempView("reviews")
spark.catalog.listTables()

[Table(name='listings', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='reviews', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [11]:
# Minh chứng số dòng > 100.000 dòng
spark.sql("SELECT COUNT(*) AS Total_rows FROM listings").show()

+----------+
|Total_rows|
+----------+
|    279712|
+----------+



In [12]:
# Minh chứng số cột >10
num_cols = len(listings_df.columns)
print(f"Tổng số cột: {num_cols}")

Tổng số cột: 33


In [13]:
# Câu truy vấn 2: Đánh giá hiệu quả của danh hiệu Superhost đối với việc thu hút khách hàng và chất lượng dịch vụ
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   CASE L.host_is_superhost
       WHEN 't' THEN 'superhost'
       ELSE 'none'
   END AS host_is_superhost,
   COUNT(DISTINCT L.listing_id) AS total_listings,
   COUNT(R.review_id) AS reviews,
   ROUND(AVG(L.host_response_rate)*100, 2) AS avg_host_reponse,
   ROUND(AVG(L.host_acceptance_rate)*100, 2) AS avg_host_acceptance,
   ROUND(AVG(L.review_scores_rating), 2) AS avg_rating,
   ROUND(AVG(L.review_scores_cleanliness), 2) AS avg_cleanliness,
   ROUND(AVG(L.review_scores_communication), 2) AS avg_communication,
   ROUND(AVG(L.review_scores_value), 2) AS avg_value
FROM listings L
JOIN reviews R ON L.listing_id = R.listing_id
WHERE L.host_is_superhost IS NOT NULL
GROUP BY L.host_is_superhost
ORDER BY L.host_is_superhost ASC;""").toPandas()

[USER: Pham Hong Nhu] [HOST: LAPTOP-K48F5AJQ] [INFO]: Query execution started...


,host_is_superhost,total_listings,reviews,avg_host_reponse,avg_host_acceptance,avg_rating,avg_cleanliness,avg_communication,avg_value
0,none,147904,3009692,89.88,87.70,92.80,9.27,9.72,9.27
1,superhost,45540,2359512,97.24,93.37,96.85,9.79,9.97,9.73


In [14]:
# Câu truy vấn 3: Cơ cấu nguồn cung phòng và giá thuê theo quy mô sức chứa khách tại từng thành phố
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   city,
   CASE
       WHEN accommodates <= 2 THEN '1-2 khách (Cá nhân/Cặp đôi)'
       WHEN accommodates BETWEEN 3 AND 5 THEN '3-5 khách (Gia đình/Nhóm nhỏ)'
       ELSE 'Trên 5 khách (Đoàn du lịch)'
   END AS room_capacity_segment,
   COUNT(listing_id) AS total_rooms_listed,
   ROUND(AVG(price_usd), 2) AS avg_nightly_price
FROM listings
WHERE accommodates IS NOT NULL
GROUP BY city,
   CASE
       WHEN accommodates <= 2 THEN '1-2 khách (Cá nhân/Cặp đôi)'
       WHEN accommodates BETWEEN 3 AND 5 THEN '3-5 khách (Gia đình/Nhóm nhỏ)'
       ELSE 'Trên 5 khách (Đoàn du lịch)'
   END
ORDER BY city,room_capacity_segment ASC;""").toPandas()

[USER: Pham Hong Nhu] [HOST: LAPTOP-K48F5AJQ] [INFO]: Query execution started...


,city,room_capacity_segment,total_rooms_listed,avg_nightly_price
0,Bangkok,1-2 khách (Cá nhân/Cặp đôi),10976,48.26
1,Bangkok,3-5 khách (Gia đình/Nhóm nhỏ),6288,64.54
2,Bangkok,Trên 5 khách (Đoàn du lịch),2097,138.71
3,Cape Town,1-2 khách (Cá nhân/Cặp đôi),8769,60.74
4,Cape Town,3-5 khách (Gia đình/Nhóm nhỏ),5814,97.71
5,Cape Town,Trên 5 khách (Đoàn du lịch),4503,380.47
6,Hong Kong,1-2 khách (Cá nhân/Cặp đôi),4912,69.18
7,Hong Kong,3-5 khách (Gia đình/Nhóm nhỏ),1486,132.99
8,Hong Kong,Trên 5 khách (Đoàn du lịch),689,199.31
9,Istanbul,1-2 khách (Cá nhân/Cặp đôi),12417,8.47


In [9]:
#Câu truy vấn 4: Truy vấn phân chia phòng theo các phân khúc giá (Giá rẻ, Trung bình, Cao cấp) và tính điểm đánh giá trung bình (avg_rating) của từng phân khúc
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""
SELECT
   CASE
       WHEN CAST(price AS DOUBLE) < 50 THEN '1. Gia re'
       WHEN CAST(price AS DOUBLE) BETWEEN 50 AND 150 THEN '2. Trung binh'
       ELSE '3. Cao cap'
   END AS price_tier,
   ROUND(AVG(CAST(review_scores_rating AS DOUBLE)), 2) AS avg_rating,
   COUNT(*) AS total_listings
FROM
   listings
WHERE
   price IS NOT NULL
   AND review_scores_rating IS NOT NULL
GROUP BY
   1
ORDER BY
   price_tier ASC;
""").toPandas()


[USER: DELL] [HOST: LE-VIET-BAO] [INFO]: Query execution started...


,price_tier,avg_rating,total_listings
0,1. Gia re,91.99,19577
1,2. Trung binh,93.31,83211
2,3. Cao cap,93.82,85519


In [10]:
#Câu truy vấn 5: Truy vấn thống kê top 20 tiện ích xuất hiện phổ biến nhất trên toàn bộ hệ thống
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""
 SELECT
  exploded_amenity AS amenity_name,
  COUNT(*) AS tan_suat
 FROM listings
 LATERAL VIEW EXPLODE(amenities) AS exploded_amenity
 GROUP BY exploded_amenity
 ORDER BY tan_suat DESC LIMIT 20
""").toPandas()

[USER: DELL] [HOST: LE-VIET-BAO] [INFO]: Query execution started...


,amenity_name,tan_suat
0,Wifi,260090
1,Essentials,253532
2,Long term stays allowed,241054
3,Kitchen,240923
4,TV,213037
5,Hangers,211356
6,Hair dryer,188724
7,Iron,187756
8,Washer,185073
9,Heating,184327


In [11]:
#Câu truy vấn 6: Truy vấn tính toán tính ra tỷ trọng phần trăm của từng loại phòng trong mỗi khu vực
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql ("""
SELECT
   neighbourhood,
   room_type,
   COUNT(*) AS total_listings,
   ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY neighbourhood), 2) AS pct_share_in_area
FROM listings
GROUP BY neighbourhood, room_type
ORDER BY neighbourhood, pct_share_in_area DESC;
""").toPandas()

[USER: DELL] [HOST: LE-VIET-BAO] [INFO]: Query execution started...


,neighbourhood,room_type,total_listings,pct_share_in_area
0,Abolicao,Entire place,4,80.00
1,Abolicao,Shared room,1,20.00
2,Acari,Entire place,1,100.00
3,Adalar,Entire place,124,63.59
4,Adalar,Private room,60,30.77
...,...,...,...,...
1846,Zeytinburnu,Entire place,62,53.45
1847,Zeytinburnu,Private room,47,40.52
1848,Zeytinburnu,Shared room,6,5.17
1849,Zeytinburnu,Hotel room,1,0.86


In [15]:
# Câu truy vấn 7: Đánh giá các tiện ích cốt lõi thuộc phân khúc căn hộ Cao cấp
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""SELECT
   exploded_amenity AS amenity_name,
   COUNT(*) AS total_occurrence
FROM listings
LATERAL VIEW EXPLODE(amenities) AS exploded_amenity
WHERE price_usd > (SELECT AVG(price_usd) FROM listings)
GROUP BY exploded_amenity
ORDER BY total_occurrence DESC
LIMIT 5;""").toPandas()

[USER: Pham Hong Nhu] [HOST: LAPTOP-K48F5AJQ] [INFO]: Query execution started...


,amenity_name,total_occurrence
0,Wifi,66128
1,Essentials,62925
2,Kitchen,62348
3,Long term stays allowed,60150
4,TV,59949


In [12]:

#Câu truy vấn 8: Truy vấn khoanh vùng các khu vực kinh doanh trọng điểm trong mười hai tháng gần nhất  và trích xuất bộ chỉ số hiệu năng của ba căn hộ tối ưu nhất nội vùng làm đường cơ sở tiêu chuẩn.
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""
WITH date_anchor AS (
   -- Bước 1: Tìm ngày đánh giá mới nhất thực tế có trong tập dữ liệu để làm mốc
   SELECT CAST(MAX(date) AS DATE) AS max_date FROM reviews
),
ltm_reviews_by_listing AS (
   -- Bước 2: Tính tổng số review LTM dựa trên ngày mốc vừa tìm được
   SELECT r.listing_id,
       COUNT(r.review_id) AS total_reviews_ltm
   FROM reviews r
   CROSS JOIN date_anchor a
   WHERE r.date >= CAST(DATE_SUB(a.max_date, 365) AS STRING)
   GROUP BY r.listing_id
   HAVING COUNT(r.review_id) > 0
),
ranked_listings AS (
   -- Bước 3: JOIN với bảng listings theo đúng trường 'listing_id'
   SELECT l.neighbourhood,l.listing_id,l.name,l.price,r.total_reviews_ltm,
       l.review_scores_cleanliness,l.review_scores_communication,
       ROW_NUMBER() OVER (PARTITION BY l.neighbourhood ORDER BY r.total_reviews_ltm DESC) AS hang_trong_vung
   FROM ltm_reviews_by_listing r
   JOIN listings l ON r.listing_id = l.listing_id ),
top_neighbourhoods AS (
   -- Bước 4: Lấy ra top các khu vực có tổng review LTM cao nhất
   SELECT neighbourhood,
       SUM(total_reviews_ltm) AS tong_reviews_khu_vuc
   FROM ranked_listings
   GROUP BY neighbourhood
   ORDER BY tong_reviews_khu_vuc DESC
   LIMIT 10
)
-- Bước 5: Xuất kết quả chi tiết
SELECT l.neighbourhood,l.listing_id,l.name,l.price,l.total_reviews_ltm,
   l.review_scores_cleanliness,l.review_scores_communication
FROM ranked_listings l
JOIN top_neighbourhoods n ON l.neighbourhood = n.neighbourhood
WHERE l.hang_trong_vung <= 3
ORDER BY n.tong_reviews_khu_vuc DESC,l.total_reviews_ltm DESC;
""").toPandas()


[USER: DELL] [HOST: LE-VIET-BAO] [INFO]: Query execution started...


,neighbourhood,listing_id,name,price,total_reviews_ltm,review_scores_cleanliness,review_scores_communication
0,Cuauhtemoc,43246311,Habitacion para estancias largas,518,194,10.0,10.0
1,Cuauhtemoc,45175157,Habitacion Privada en el Corazon de la Roma,642,158,9.0,10.0
2,Cuauhtemoc,46547879,Habitacion privada en el Zocalo de la Ciudad,651,131,10.0,10.0
3,I Centro Storico,32185249,Romantic suite in Campo de Fiori,91,110,10.0,10.0
4,I Centro Storico,23880958,REAL BEST VIEW COLISEUM LUXURY LOFT HISTORY CE...,68,75,10.0,10.0
5,I Centro Storico,17680772,Valentine's Day Special Offer,30,73,10.0,10.0
6,Beyoglu,21811046,Well Designed Studio Apartment in BEYOGLU,252,181,9.0,10.0
7,Beyoglu,32186550,@KarakÃ¶y Nice Big Studio Apartment,171,141,10.0,10.0
8,Beyoglu,34331518,Zen Apartments 4,144,84,9.0,10.0
9,Copacabana,35386228,COPACABANA LUXO & DESIGN SELF CHECK IN 24 HORAS,260,114,10.0,10.0


In [10]:

# QUERY 1 – Diem hanh vi tot & ty le Superhost
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""
  SELECT diem_hanh_vi_tot,
         COUNT(*)                       AS so_listing,
         ROUND(100.0*AVG(la_super), 2)  AS ty_le_superhost
  FROM (
      SELECT CASE WHEN host_is_superhost = 't' THEN 1 ELSE 0 END AS la_super,
             ( CASE WHEN host_response_time   = 'within an hour' THEN 1 ELSE 0 END
             + CASE WHEN host_identity_verified = 't'            THEN 1 ELSE 0 END
             + CASE WHEN instant_bookable       = 't'            THEN 1 ELSE 0 END ) AS diem_hanh_vi_tot
      FROM listings
  )
  GROUP BY diem_hanh_vi_tot
  ORDER BY diem_hanh_vi_tot
""").toPandas()

[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,diem_hanh_vi_tot,so_listing,ty_le_superhost
0,0,40076,5.97
1,1,118466,12.59
2,2,81714,23.74
3,3,39456,34.33


In [11]:
# Câu truy vấn số 9 Làm thế nào để nhận diện sớm các Superhost đang có nguy cơ tụt hạng trước kỳ đánh giá tiếp theo của Airbnb?
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""
WITH host_agg AS (
 SELECT host_id, FIRST(city) AS city, COUNT(*) AS n_listing,
        ROUND(AVG(review_scores_rating),2)  AS rating_tb,
        MAX(host_is_superhost) AS is_super
 FROM listings
 WHERE review_scores_rating IS NOT NULL
 GROUP BY host_id
 HAVING MAX(host_is_superhost) = 't'
),
city_bench AS (
 SELECT city, AVG(rating_tb) AS rating_city FROM host_agg GROUP BY city
),
RankedAnalysis AS (
 SELECT h.host_id, h.city, h.n_listing, h.rating_tb,
        ROUND(h.rating_tb - c.rating_city, 2) AS lech_vs_city,
        NTILE(4) OVER (ORDER BY h.rating_tb) AS nhom_rui_ro,
        CASE WHEN h.rating_tb < c.rating_city THEN 'SUPERHOST RUI RO'
             ELSE 'OK' END AS canh_bao
 FROM host_agg h JOIN city_bench c ON h.city = c.city
 WHERE h.n_listing >= 2
)
SELECT host_id, city, n_listing, rating_tb, lech_vs_city, nhom_rui_ro, canh_bao
FROM (
 SELECT *,
        ROW_NUMBER() OVER (PARTITION BY nhom_rui_ro ORDER BY lech_vs_city ASC) AS thut_tu_trong_nhom
 FROM RankedAnalysis
)
WHERE thut_tu_trong_nhom <= 5
ORDER BY nhom_rui_ro ASC, lech_vs_city ASC
""").toPandas()

[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,host_id,city,n_listing,rating_tb,lech_vs_city,nhom_rui_ro,canh_bao
0,309651408,Cape Town,2,50.00,-47.68,1,SUPERHOST RUI RO
1,125422956,Sydney,2,58.50,-39.16,1,SUPERHOST RUI RO
2,31168193,Cape Town,2,70.00,-27.68,1,SUPERHOST RUI RO
3,152690866,Istanbul,5,71.00,-26.28,1,SUPERHOST RUI RO
4,274418925,Bangkok,3,71.67,-25.40,1,SUPERHOST RUI RO
5,278252599,Mexico City,2,96.00,-1.68,2,SUPERHOST RUI RO
6,271659755,Mexico City,2,96.00,-1.68,2,SUPERHOST RUI RO
7,239389564,Mexico City,3,96.00,-1.68,2,SUPERHOST RUI RO
8,211773688,Mexico City,3,96.00,-1.68,2,SUPERHOST RUI RO
9,196839762,Mexico City,2,96.00,-1.68,2,SUPERHOST RUI RO


In [12]:
# Câu truy vấn số 10 Tỷ lệ Superhost phân bố như thế nào giữa các loại hình phòng tại từng thành phố?
print(f"{get_system_context()} [INFO]: Query execution started...")
spark.sql("""
SELECT * FROM (
SELECT city, room_type,
       CASE WHEN host_is_superhost='t' THEN 1.0 ELSE 0.0 END AS is_super
FROM listings
) PIVOT (
ROUND(AVG(is_super)*100, 1)
FOR room_type IN ('Entire place' AS entire, 'Private room' AS private,
                  'Hotel room' AS hotel, 'Shared room' AS shared)
)
""").toPandas()

[USER: Quynh Nga] [HOST: DESKTOP-TC9PES0] [INFO]: Query execution started...


,city,entire,private,hotel,shared
0,Sydney,13.8,9.7,22.7,2.5
1,Mexico City,39.8,23.1,25.2,18.9
2,Cape Town,27.6,14.5,11.4,7.1
3,Paris,12.0,15.7,20.3,8.5
4,Istanbul,18.0,7.8,14.8,6.0
5,Hong Kong,14.6,21.7,3.9,13.7
6,Rome,30.4,18.4,18.4,10.0
7,Rio de Janeiro,18.7,13.1,26.3,8.7
8,Bangkok,25.7,12.6,16.1,15.0
9,New York,20.0,17.6,14.4,15.7


In [ ]:
u27t8a58